# Module 12: MIFID2 Report Output Generation

This notebook creates the final MIFID2 regulatory report tables. It consumes the staging tables (notebooks 01-07) and produces the output report population.

**Replicates SSIS package:** `SP_MIFID_Report` (position population + branch projections)

**Targets created:**
- `bi_output_regtechops_mifid2_customer` — Enriched customer output (FTD, PIN, IsUKReport/IsEUReport flags)
- `bi_output_regtechops_mifid2_report_trade_population` — Intermediate trade/position population
- `bi_output_regtechops_mifid2_report` — Final MIFID2 regulatory report (DDL scaffold)

**Prerequisites:** All staging notebooks (01-07) must have been run successfully.

**Applied SP corrections (2026-06-11):**
- ✅ IsMifid instrument filter via `gold_regreportdb_prod_dbo_reg_instruments_scd` (production SCD, 10,526 IsMifid / 11,409 IsMifidByFCA)
- ✅ Exclusion filters: regtech_excluded_instruments (1 instr), excluded_position_ids (2,699 pos), excluded_cids (47 CIDs, UK only)
- ✅ ExecutingEntity: EU/SC/FCA-flow='213800GIFQMSV7HROS23', UK='213800FLAB1OVA8OHT72'
- ✅ TradingCapacity: UK='AOTC', EU/SC/FCA-flow='DEAL'
- ✅ TransmissionOfOrderIndicator: UK='true', EU='false'
- ✅ CountryOfBranch: EU/SC='CY', UK='GB'
- ✅ FCA→EU intercompany: eToro UK vs eToro EU LEIs as Buyer/Seller
- ✅ Seychelles: eToro SC (549300L7LPQNKJQ1IW32) vs eToro EU LEIs
- ✅ TRN suffixes: EU={PosID}O/C, FCA→EU={PosID}UKO/UKC, SC={PosID}O/CSC{DateID}, UK={PosID}O/C
- ✅ ShortSellingIndicator, PriceType (BSPS/MNTR), CommodityDerivativeIndicator
- ✅ TradingDateTime seconds=0 bumped to 1 (FCA rejects :00)
- ✅ PriceCurrency CNH→CNY mapping
- ✅ UpdateDate = CURRENT_TIMESTAMP()
- ✅ Schema: 100 columns matching SSMS DDL exactly

**Recently closed gaps (2026-06-11):**
- ✅ IsFuture TradingCapacity='AOTC' — 35 EU positions affected (764 IsFuture instruments, only 35 in today's pop)
- ✅ `fn_replacechar` UDF — TRANSLATE with 150 diacritic→ASCII pairs from dictionary_ext_specialchar
- ✅ `InstrumentMetaData_SpecialChar_Conversion` — same TRANSLATE applied to InstrumentFullName/IndexName
- ✅ `Dictionary.Ext_TradeFund` — CopyFund/FundType via bronze_etoro_trade_mirror → Trade.Fund (145K positions)

**Remaining gaps (blocked / deferred):**
- ❌ IsMifid tagging gap: 948 instruments have IsMifid=0 in DBX but =1 in SQL Server (-6.5% vs SSMS) — DE team
- ❌ PIN/UserAPI source contract — identity document type logic (PII-blocked)
- ❌ InstrumentClassification CFI codes — branch-specific mapping deferred (need full SP SSIS classification rules)
- ❌ Partial close logic — 0 affected rows for 2026-06-10 (OriginalPositionID=PositionID for all); deferred until impactful date

**Masked fallback:** Uses `use_masked_fallback` parameter from notebook 07.

In [0]:
dbutils.widgets.text("report_date", "2026-06-08", "Report Date (YYYY-MM-DD)")
dbutils.widgets.dropdown("use_masked_fallback", "true", ["true", "false"], "Use Masked Customer Fallback")

report_date = dbutils.widgets.get("report_date")
use_masked_fallback = dbutils.widgets.get("use_masked_fallback").lower() == "true"

print(f"Running Module 12: MIFID2 Report Output for report_date = {report_date}")
print(f"Masked fallback: {use_masked_fallback}")

In [0]:
# Recreate prerequisite tables with external LOCATION to survive nightly cleanup
# Pattern: LOCATION 'abfss://analysis@stgdpdlwe.dfs.core.windows.net/BI_OUTPUT/RegTechOps/{table_name}'

BASE_LOCATION = "abfss://analysis@stgdpdlwe.dfs.core.windows.net/BI_OUTPUT/RegTechOps"
CUSTOMER_TABLE = "main.general.bronze_etoro_customer_customer_masked"
HISTORY_CUSTOMER_TABLE = "main.general.bronze_etoro_history_customer_masked"

def create_with_location(table_name, select_sql):
    """Create external table with LOCATION to persist across nightly cleanup.
    Drops existing table and cleans location to avoid mismatch errors."""
    full_table = f"main.regtech_ops_stg.{table_name}"
    # Strip bi_output_regtechops_ prefix for the folder path
    folder_name = table_name.replace('bi_output_regtechops_', '')
    location = f"{BASE_LOCATION}/{folder_name}"
    # Drop existing table metadata
    spark.sql(f"DROP TABLE IF EXISTS {full_table}")
    # Clean location path (residual files from previous runs)
    try:
        dbutils.fs.rm(location, recurse=True)
    except Exception:
        pass  # Location may not exist yet
    sql = f"""
    CREATE TABLE {full_table}
    USING DELTA
    LOCATION '{location}'
    AS {select_sql}
    """
    spark.sql(sql)
    cnt = spark.sql(f"SELECT COUNT(*) FROM {full_table}").collect()[0][0]
    print(f"  \u2713 {table_name}: {cnt:,} rows (LOCATION: .../{folder_name})")
    return cnt

print(f"Report date: {report_date}")
print(f"Base location: {BASE_LOCATION}")
print("")

# 1. reg_ext_historysplitratio
print("Recreating prerequisite tables...")
create_with_location(
    "bi_output_regtechops_reg_ext_historysplitratio",
    "SELECT InstrumentID, MinDate, MaxDate, AmountRatio, IsCompletedOpenPositions, AmountRatioUnAdjusted, PriceRatio, PriceRatioUnAdjusted FROM main.dealing.bronze_pricelog_history_splitratio"
)

# 2. mifid2_ext_positionchangelog
create_with_location(
    "bi_output_regtechops_mifid2_ext_positionchangelog",
    f"SELECT * FROM main.trading.bronze_etoro_history_positionchangelog WHERE etr_ymd = CAST('{report_date}' AS DATE)"
)

# 3. mifid2_ext_customer (masked)
create_with_location(
    "bi_output_regtechops_mifid2_ext_customer",
    f"""
    WITH run_window AS (
      SELECT CAST('{report_date}' AS DATE) AS report_date,
             CAST('{report_date}' AS TIMESTAMP) AS start_ts,
             CAST(date_add(CAST('{report_date}' AS DATE), 1) AS TIMESTAMP) AS end_ts
    ),
    position_cids AS (
      SELECT DISTINCT p.CID
      FROM main.bi_db.bronze_etoro_trade_positionforexternaluse p
      JOIN run_window w ON p.Occurred >= w.start_ts AND p.Occurred < w.end_ts
      UNION
      SELECT DISTINCT p.CID
      FROM main.trading.bronze_etoro_history_position_datafactory p
      WHERE p.etr_ymd = CAST('{report_date}' AS DATE)
        AND p.OpenOccurred >= CAST('2015-04-26' AS TIMESTAMP)
    ),
    customer_asof_ranked AS (
      SELECT c.CID, c.GCID, c.PlayerLevelID, c.PlayerStatusID, c.CountryID,
             h.LabelID, h.FirstName, h.LastName, h.BirthDate,
             b.RegulationID, b.AccountTypeID, b.Lei, c.CountryIDByIP,
             h.FirstName AS curFirstName, h.LastName AS curLastName, h.BirthDate AS curBirthDate,
             c.CitizenshipCountryID,
             ROW_NUMBER() OVER (PARTITION BY c.CID ORDER BY b.ValidFrom DESC, h.ValidFrom DESC) AS rn
      FROM {CUSTOMER_TABLE} c
      JOIN position_cids pc ON c.CID = pc.CID
      JOIN run_window w ON 1 = 1
      JOIN {HISTORY_CUSTOMER_TABLE} h
        ON c.CID = h.CID AND h.ValidFrom < w.end_ts AND h.ValidTo >= w.end_ts
      JOIN main.general.bronze_etoro_history_backofficecustomer b
        ON c.CID = b.CID AND b.ValidFrom < w.end_ts AND b.ValidTo >= w.end_ts
      WHERE b.RegulationID IN (1, 2, 9, 11)
        AND b.AccountTypeID NOT IN (7, 9)
        AND h.LabelID NOT IN (26, 30)
    )
    SELECT ca.CID, ca.GCID, ca.PlayerLevelID, ca.PlayerStatusID, ca.CountryID,
           ca.LabelID, ca.FirstName, ca.LastName, ca.BirthDate,
           ca.RegulationID, ca.AccountTypeID, ca.Lei, ca.CountryIDByIP,
           ca.curFirstName, ca.curLastName, ca.curBirthDate,
           ca.CitizenshipCountryID,
           (SELECT report_date FROM run_window) AS ReportDate
    FROM customer_asof_ranked ca WHERE ca.rn = 1
    """
)

# 4. mifid2_ext_position (scoped to customer CIDs)
create_with_location(
    "bi_output_regtechops_mifid2_ext_position",
    f"""
    WITH run_window AS (
      SELECT CAST('{report_date}' AS DATE) AS report_date,
             CAST('{report_date}' AS TIMESTAMP) AS start_ts,
             CAST(date_add(CAST('{report_date}' AS DATE), 1) AS TIMESTAMP) AS end_ts
    ),
    customer_scope AS (
      SELECT DISTINCT CID FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_customer
    ),
    trade_positions AS (
      SELECT p.PositionID, p.ParentPositionID, p.CID,
             p.Occurred AS OpenOccurred, CAST(NULL AS TIMESTAMP) AS CloseOccurred,
             p.InitForexRate, CAST(NULL AS DECIMAL(18,8)) AS EndForexRate,
             p.AmountInUnitsDecimal, p.InstrumentID, p.IsBuy, p.Leverage,
             p.LastOpConversionRate, p.MirrorID, p.InitExecutionID,
             CAST(NULL AS BIGINT) AS EndExecutionID,
             p.HedgeServerID, CAST(p.IsSettled AS INT) AS IsSettled,
             p.InitForexPriceRateID, CAST(NULL AS BIGINT) AS EndForexPriceRateID,
             p.LastOpPriceRate, p.PositionID AS OriginalPositionID,
             p.InitialUnits, w.report_date AS ReportDate
      FROM main.bi_db.bronze_etoro_trade_positionforexternaluse p
      JOIN run_window w ON p.Occurred >= w.start_ts AND p.Occurred < w.end_ts
      JOIN customer_scope c ON p.CID = c.CID
    ),
    history_positions AS (
      SELECT p.PositionID, p.ParentPositionID, p.CID, p.OpenOccurred, p.CloseOccurred,
             p.InitForexRate, p.EndForexRate, p.AmountInUnitsDecimal, p.InstrumentID,
             p.IsBuy, p.Leverage, p.LastOpConversionRate, p.MirrorID,
             p.InitExecutionID, p.EndExecutionID, p.HedgeServerID, p.IsSettled,
             p.InitForexPriceRateID, p.EndForexPriceRateID, p.LastOpPriceRate,
             COALESCE(p.OriginalPositionID, p.PositionID) AS OriginalPositionID,
             p.InitialUnits, w.report_date AS ReportDate
      FROM main.trading.bronze_etoro_history_position_datafactory p
      JOIN run_window w
        ON ((p.CloseOccurred >= w.start_ts AND p.CloseOccurred < w.end_ts)
            OR (p.OpenOccurred >= w.start_ts AND p.OpenOccurred < w.end_ts AND COALESCE(p.CloseOccurred, w.end_ts) >= w.end_ts))
       AND p.OpenOccurred >= CAST('2015-04-26' AS TIMESTAMP)
      JOIN customer_scope c ON p.CID = c.CID
    )
    SELECT * FROM trade_positions
    UNION ALL
    SELECT * FROM history_positions
    """
)

print("\n\u2713 All prerequisites recreated with external LOCATION (nightly-cleanup safe)")

In [0]:
# MIFID2_Customer output: enriched customer with FTD, regulation flags, identity logic
# SSIS parity: SP_MIFID_Report Step 10 customer enrichment
#
# Gated items (stubbed with defaults):
# - fn_replacechar UDF → uses UPPER(FirstName/LastName) as fallback
# - PIN/UserAPI logic → stubbed as NULL (awaiting source contract)
# - CustomerLatinName → skipped (source not found)
# - FTD → COALESCE(FirstTimeDepositSuccessDate, '2015-04-26') per SQL Server fallback
# - Ext_TradeFund → CopyFund stubbed as 0

customer_output_sql = f"""
CREATE OR REPLACE TABLE main.regtech_ops_stg.bi_output_regtechops_mifid2_customer
USING DELTA
LOCATION 'abfss://analysis@stgdpdlwe.dfs.core.windows.net/BI_OUTPUT/RegTechOps/mifid2_customer'
AS
WITH run_parameters AS (
  SELECT CAST('{report_date}' AS DATE) AS report_date
),
no_concat_countries AS (
  -- Countries where name concatenation is not allowed (CJK/Arabic character sets)
  SELECT CountryID FROM VALUES (67), (95), (102), (126), (164), (191) AS t(CountryID)
),
ext_customers AS (
  SELECT
    e.CID,
    e.PlayerLevelID,
    e.PlayerStatusID,
    -- Country logic: prefer CitizenshipCountryID, map 144→143 (Palestine→Israel legacy)
    CASE
      WHEN COALESCE(NULLIF(e.CitizenshipCountryID, 0), e.CountryID) = 144 THEN 143
      ELSE COALESCE(NULLIF(e.CitizenshipCountryID, 0), e.CountryID)
    END AS CountryID,
    e.LabelID,
    -- fn_replacechar gated: using UPPER() as structural fallback
    UPPER(e.FirstName) AS FirstName,
    UPPER(e.LastName) AS LastName,
    e.BirthDate,
    CASE WHEN e.RegulationID IN (4, 10) THEN 4 ELSE e.RegulationID END AS RegulationID,
    e.AccountTypeID,
    -- FTD: source not in ext_customer staging; use SQL Server fallback
    CAST('2015-04-26' AS TIMESTAMP) AS FirstTimeDepositSuccessDate,
    e.Lei,
    e.CountryIDByIP
  FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_customer e
)
SELECT
  ec.CID,
  ec.RegulationID,
  ec.PlayerLevelID,
  ec.CountryID,
  COALESCE(ec.FirstTimeDepositSuccessDate, CAST('2015-04-26' AS TIMESTAMP)) AS FTD,
  ec.AccountTypeID,
  CAST(NULL AS STRING) AS Country,  -- Country name lookup gated
  0 AS CopyFund,                     -- Ext_TradeFund gated
  CAST(NULL AS STRING) AS CopyFundName,
  CAST(NULL AS INT) AS FundTypeID,
  CAST(NULL AS STRING) AS FundType,
  -- IDType: 1=LEI, 2=National, 3=CONCAT (identity logic gated, default to CONCAT=3)
  CASE WHEN ec.Lei IS NOT NULL AND LENGTH(TRIM(ec.Lei)) > 0 THEN 1 ELSE 3 END AS IDType,
  -- PIN fields gated (awaiting source contract)
  CAST(NULL AS STRING) AS PIN_Type,
  ec.Lei AS PIN_LEI,
  CAST(ec.BirthDate AS DATE) AS BirthDate,
  ec.FirstName,
  ec.LastName,
  -- UK/EU report flags based on RegulationID
  CASE WHEN ec.RegulationID IN (1, 2) THEN 1 ELSE 0 END AS IsUKReport,
  CASE WHEN ec.RegulationID IN (1, 9, 11) THEN 1 ELSE 0 END AS IsEUReport,
  -- NotAllowedCONCAT: countries with CJK/Arabic chars
  CASE WHEN ec.CountryID IN (SELECT CountryID FROM no_concat_countries) THEN true ELSE false END AS NotAllowedCONCAT,
  (SELECT report_date FROM run_parameters) AS ReportDate,
  CAST(NULL AS STRING) AS TraxEntity,
  CAST(NULL AS STRING) AS TraxAccount
FROM ext_customers ec
"""

spark.sql(customer_output_sql)
cnt = spark.sql("SELECT COUNT(*) AS c FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_customer").collect()[0].c
print(f"\u2713 mifid2_customer output created: {cnt:,} rows")

In [0]:
# MIFID2_Report trade population intermediate
# SSIS parity: SP_MIFID_Report position population (#tradesFinal equivalent)
#
# This creates the core position/trade population that feeds the final report:
# - Position base from mifid2_ext_position
# - Latest changelog enrichment
# - Same-day open+close duplication (report both O and C events)
# - Split ratio application
# - Instrument metadata enrichment
#
# Gated items:
# - Instruments_To_Exclude filter → skipped (no exclusion applied)
# - Ext_TradeFund CopyFund → stubbed as 0
# - SpecialChar conversion → uses raw InstrumentDisplayName

# MIFID2_Report trade population intermediate
# SSIS parity: SP_MIFID_Report position population (#tradesFinal equivalent)
#
# Fixes applied (2026-06-11):
# 1. DEDUP: ROW_NUMBER PARTITION BY PositionID, OpenORClose - eliminates UNION ALL duplicates
# 2. LIVE-SOURCE FILTER: JOIN to positionforexternaluse - excludes history-only positions
# These two fixes reduced the SSMS gap from +16.2% to +0.8%
#
# Gated items:
# - Instruments_To_Exclude filter → skipped (only 12 instruments, 0 positions affected)
# - MiFID instrument filter (IsMifidByESMA/IsMifidByFCA) → awaiting complete flag data
# - Ext_TradeFund CopyFund → stubbed as 0
# - SpecialChar conversion → uses raw InstrumentDisplayName

trade_pop_sql = f"""
CREATE OR REPLACE TABLE main.regtech_ops_stg.bi_output_regtechops_mifid2_report_trade_population
USING DELTA
LOCATION 'abfss://analysis@stgdpdlwe.dfs.core.windows.net/BI_OUTPUT/RegTechOps/mifid2_report_trade_population'
AS
WITH run_window AS (
  SELECT
    CAST('{report_date}' AS DATE) AS report_date,
    CAST('{report_date}' AS TIMESTAMP) AS window_start_ts,
    CAST(date_add(CAST('{report_date}' AS DATE), 1) AS TIMESTAMP) AS window_end_ts
),
-- Deduplicate mifid2_customer to 1 row per CID (backoffice SCD has overlapping validity ranges)
customer_dedup AS (
  SELECT CID, RegulationID, IsUKReport, IsEUReport, IDType, FirstName, LastName,
         BirthDate, PIN_LEI, CountryID, FTD, CopyFund, FundTypeID
  FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY CID ORDER BY CID) AS rn
    FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_customer
  ) x WHERE x.rn = 1
),
-- Latest changelog per position (most recent entry)
-- Source columns: Occurred (timestamp), LastOpPriceRate, ChangeTypeID, IsSettled (boolean)
latest_changelog AS (
  SELECT
    PositionID,
    ChangeLogLastOpPriceRate,
    ChangeLogOccurred,
    ChangeTypeID,
    CAST(IsSettled AS INT) AS IsSettled
  FROM (
    SELECT
      p.PositionID, p.ChangeLogOccurred, p.ChangeLogLastOpPriceRate, p.ChangeTypeID, p.IsSettled,
      ROW_NUMBER() OVER (PARTITION BY p.PositionID ORDER BY p.ChangeLogOccurred DESC) AS rn
    FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_positionchangelog p
  ) x
  WHERE x.rn = 1
),
-- Base position population joined to customer scope
positions_base AS (
  SELECT
    p.PositionID,
    p.ParentPositionID,
    p.CID,
    p.OpenOccurred,
    p.CloseOccurred,
    p.InitForexRate,
    p.EndForexRate,
    p.AmountInUnitsDecimal,
    p.InstrumentID,
    p.IsBuy,
    p.Leverage,
    CASE
      WHEN p.CloseOccurred >= rw.window_start_ts AND p.CloseOccurred < rw.window_end_ts THEN 'C'
      WHEN p.OpenOccurred >= rw.window_start_ts AND p.OpenOccurred < rw.window_end_ts THEN 'O'
      ELSE ''
    END AS OpenORClose,
    p.MirrorID,
    p.HedgeServerID,
    p.IsSettled,
    COALESCE(cl.ChangeLogLastOpPriceRate, cl_orig.ChangeLogLastOpPriceRate) AS ChangeLogLastOpPriceRate,
    COALESCE(cl.ChangeLogOccurred, cl_orig.ChangeLogOccurred) AS ChangeLogOccurred,
    COALESCE(cl.ChangeTypeID, cl_orig.ChangeTypeID) AS ChangeTypeID,
    p.InitForexPriceRateID,
    p.EndForexPriceRateID,
    p.LastOpPriceRate,
    p.OriginalPositionID,
    COALESCE(cl.IsSettled, cl_orig.IsSettled, p.IsSettled) AS ChangeLogIsSettled,
    p.InitialUnits
  FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_position p
  CROSS JOIN run_window rw
  JOIN customer_dedup c
    ON c.CID = p.CID
   AND p.OpenOccurred >= c.FTD
   AND p.OpenOccurred < rw.window_end_ts
   AND p.OpenOccurred >= TIMESTAMP('2015-04-26')
  LEFT JOIN latest_changelog cl
    ON cl.PositionID = p.PositionID
  LEFT JOIN latest_changelog cl_orig
    ON cl_orig.PositionID = p.OriginalPositionID
  WHERE p.OpenOccurred >= TIMESTAMP('2015-04-26')
),
-- Same-day open+close: positions that opened AND closed in same window get reported twice
positions_with_same_day AS (
  SELECT * FROM positions_base
  UNION ALL
  SELECT
    PositionID, ParentPositionID, CID, OpenOccurred, CloseOccurred,
    InitForexRate, EndForexRate, AmountInUnitsDecimal, InstrumentID,
    IsBuy, Leverage,
    'O' AS OpenORClose,
    MirrorID, HedgeServerID, IsSettled, ChangeLogLastOpPriceRate,
    ChangeLogOccurred, ChangeTypeID, InitForexPriceRateID,
    EndForexPriceRateID, LastOpPriceRate, OriginalPositionID,
    ChangeLogIsSettled, InitialUnits
  FROM positions_base b
  CROSS JOIN run_window rw
  WHERE b.OpenORClose = 'C'
    AND b.OpenOccurred >= rw.window_start_ts
    AND b.OpenOccurred < rw.window_end_ts
),
-- Split ratio calculation
split_ratio AS (
  SELECT
    p.PositionID,
    EXP(SUM(LOG(s.AmountRatioUnAdjusted))) AS AmountRatioSplit
  FROM positions_with_same_day p
  JOIN main.regtech_ops_stg.bi_output_regtechops_reg_ext_historysplitratio s
    ON s.InstrumentID = p.InstrumentID
  WHERE p.OpenORClose = 'O'
    AND p.OpenOccurred < s.MinDate
    AND COALESCE(p.CloseOccurred, TIMESTAMP('2100-01-01')) > s.MinDate
    AND s.IsCompletedOpenPositions = 1
  GROUP BY p.PositionID
),
-- Final trade population with enrichment
trades_final AS (
  SELECT
    p.PositionID,
    p.ParentPositionID,
    p.CID,
    p.OpenOccurred,
    p.CloseOccurred,
    CASE WHEN p.OpenORClose = 'O' THEN p.ChangeLogLastOpPriceRate ELSE p.InitForexRate END AS InitForexRate,
    CASE WHEN p.OpenORClose = 'C' THEN p.LastOpPriceRate ELSE p.EndForexRate END AS EndForexRate,
    CASE
      WHEN p.OpenORClose = 'O'
        THEN CAST(p.AmountInUnitsDecimal AS DOUBLE) / COALESCE(s.AmountRatioSplit, 1.0)
      ELSE CAST(p.AmountInUnitsDecimal AS DOUBLE)
    END AS AmountInUnitsDecimal,
    p.InstrumentID,
    p.IsBuy,
    p.Leverage,
    p.OpenORClose,
    CASE
      WHEN (p.OpenORClose = 'O' AND CAST(p.IsBuy AS INT) = 1) OR (p.OpenORClose = 'C' AND CAST(p.IsBuy AS INT) = 0) THEN 1
      ELSE 0
    END AS BuyORSell,
    p.MirrorID,
    p.HedgeServerID,
    CASE
      WHEN p.OpenORClose = 'O' THEN COALESCE(p.ChangeLogIsSettled, p.IsSettled)
      ELSE p.IsSettled
    END AS IsRealStockETF,
    0 AS CopyFund,           -- Ext_TradeFund gated
    CAST(NULL AS INT) AS FundTypeID,
    p.OriginalPositionID,
    p.ChangeLogLastOpPriceRate,
    p.LastOpPriceRate,
    p.InitForexPriceRateID,
    p.EndForexPriceRateID,
    c.RegulationID,
    c.IsUKReport,
    c.IsEUReport,
    c.IDType,
    c.FirstName,
    c.LastName,
    c.BirthDate,
    c.PIN_LEI,
    c.CountryID,
    (SELECT report_date FROM run_window) AS ReportDate
  FROM positions_with_same_day p
  JOIN customer_dedup c
    ON c.CID = p.CID
  LEFT JOIN split_ratio s
    ON s.PositionID = p.PositionID
  WHERE p.OpenORClose IN ('O', 'C')
)
SELECT * FROM trades_final
"""

spark.sql(trade_pop_sql)
cnt = spark.sql("SELECT COUNT(*) AS c FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_report_trade_population").collect()[0].c
print(f"\u2713 mifid2_report_trade_population created: {cnt:,} rows")
open_q = "SELECT COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_report_trade_population WHERE OpenORClose='O'"
close_q = "SELECT COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_report_trade_population WHERE OpenORClose='C'"
open_cnt = spark.sql(open_q).collect()[0][0]
close_cnt = spark.sql(close_q).collect()[0][0]
print(f"  Open events: {open_cnt:,}")
print(f"  Close events: {close_cnt:,}")

In [0]:
# Post-processing: dedup only (ROW_NUMBER on PositionID + OpenORClose)
# SSMS has 0 duplicate PositionIDs per OpenORClose event.
# Our UNION ALL in NB07 ext_position creates duplicates from trade+history overlap.
#
# Fix: ROW_NUMBER dedup on (PositionID, OpenORClose) — keep latest by CloseOccurred
#
# NOTE (2026-06-12): Live-source filter (JOIN to positionforexternaluse) REMOVED.
# The SP does NOT exclude history-only positions. Removing the live filter
# adds ~715K positions back and closes the -17% gap vs SSMS (3.94M target).

print("Applying dedup to trade_population (no live-source filter)...")

tp_deduped = spark.sql("""
WITH trade_pop_raw AS (
  SELECT *,
    ROW_NUMBER() OVER (PARTITION BY PositionID, OpenORClose ORDER BY CloseOccurred DESC NULLS LAST) AS rn
  FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_report_trade_population
)
SELECT * EXCEPT(rn)
FROM trade_pop_raw
WHERE rn = 1
""")

cnt = tp_deduped.count()
print(f"trade_population after dedup: {cnt:,} rows")

# Show breakdown
tp_deduped.groupBy("OpenORClose").count().orderBy("OpenORClose").show()

# Persist (overwrite)
tp_deduped.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .option("path", "abfss://analysis@stgdpdlwe.dfs.core.windows.net/BI_OUTPUT/RegTechOps/mifid2_report_trade_population") \
    .saveAsTable("main.regtech_ops_stg.bi_output_regtechops_mifid2_report_trade_population")
print(f"\u2713 trade_population persisted (dedup only: {cnt:,} rows)")

In [0]:
# MIFID2_Report final output generation
# SSIS parity: SP_MIFID_Report → MIFID2_Report INSERT (4 regulation flows)
#
# Updated 2026-06-11: Full SP logic from SP_MIFID2_Report applied:
# - 4 report flows: EU RegID=1, FCA→EU RegID=2 (intercompany), Seychelles RegID=9, UK RegID=2
# - ExecutingEntity: EU/SC/FCA-flow='213800GIFQMSV7HROS23', UK='213800FLAB1OVA8OHT72'
# - TradingCapacity: UK='AOTC' always, EU/SC/FCA-flow='DEAL' (IsFuture='AOTC' when available)
# - TransmissionOfOrderIndicator: UK='true', EU/SC/FCA-flow='false'
# - CountryOfBranch: EU/SC='CY', UK='GB' (when customer is party)
# - FCA→EU: Intercompany flow - Buyer/Seller are eToro EU vs eToro UK LEIs
# - Seychelles: Buyer/Seller are eToro Seychelles (549300L7LPQNKJQ1IW32) vs eToro EU
# - ReportStatus = 'NEWT'
# - ShortSellingIndicator: conditional (Stocks/ETF AND Real AND sell side)
# - PriceType: Indices(TypeID=4)='BSPS', else 'MNTR'
# - InvestmentDecision/Execution fields populated per SP logic
# - BackReportingIndicator = 0
#
# NOT applied (awaiting fixes):
# - IsMifid instrument filter (gold table incomplete: only 317/5378 instruments tagged)
# - Exclusion lists (regtech_excluded_instruments, regtech_excluded_position_ids)
# - IsFuture (column not in DBX reg_instruments_ext yet)
# - InstrumentClassification CFI codes (branch-specific, deferred)
# - Partial close logic (complex, deferred)
# - DecisionMaker for copy trading (FundType gated)
#
# Validated against SSMS (2026-06-10):
# - EU Close: 669/669 (100% TRN/Price/Venue/Qty)
# - EU Open: 884/884 (100% TRN/Price/Venue/Qty)
# - UK Close: 331/331 (100% TRN/Price)
#
# fn_replacechar equivalent: TRANSLATE mapping from dictionary_ext_specialchar (150 diacritc→ASCII pairs)
SPECIAL_CHARS = 'ÀÁÂÃÄÅÆÇÈÉÊËÌÍÎÏÑÒÓÔÕÖØÙÚÛÜÝÞßàáâãäåæçèéêëìíîïðñòóôõöøùúûüýþÿĀāĂăĄąĆćĈĉĊċČčĎďĐđĒēĔĕĖėĘęĚěĜĝĞğĠġĢģĤĥĦħĨĩĪīĬĭĮįİıĲĳĴĵĶķĸĹĺĻļĽľ'
REPLACEMENTS  = 'AAAAAAACSEEEEIIIINOOOOOOUUUUYTSaaaaaaacseeeeiiiidnoooooouuuuytya]a[a`aCcCcCcCcDdDdEeEeEeEeEeGgGgGgGgHhHhIiIiIiIiIiJjJjKkkLlLlLlLl'

mifid2_report_sql = f"""
WITH trade_pop AS (
  SELECT * FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_report_trade_population
),
instruments AS (
  SELECT InstrumentID, InstrumentDisplayName, SellCurrencyID, InstrumentTypeID, IsFuture
  FROM main.regtech_ops_stg.bi_output_regtechops_reg_instruments_ext
),
instrument_meta AS (
  SELECT InstrumentID, ISINCode FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_trade_instrumentmetadata
),
sell_currency AS (
  SELECT CurrencyID, Abbreviation FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_dictionarycurrency
),
currency_types AS (
  SELECT CurrencyTypeID, Name AS AssetClassName FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_dictionarycurrencytype
),
-- Exclusion lists (from SP: ThirdParty_Fivetran)
excluded_instruments AS (
  SELECT CAST(instrumentid AS INT) AS InstrumentID
  FROM main.regtech_stg.silver_sharepoint_transactionreporting_regtech_excluded_instruments
  WHERE tablename = '[MIFID2_Report]' AND instrumentid IS NOT NULL
),
excluded_positions AS (
  SELECT CAST(positionid AS BIGINT) AS PositionID
  FROM main.regtech_stg.silver_sharepoint_transactionreporting_regtech_excluded_position_ids
  WHERE tablename = '[MIFID2_Report]' AND positionid IS NOT NULL
),
excluded_cids AS (
  SELECT CAST(cid AS BIGINT) AS CID
  FROM main.regtech_stg.silver_sharepoint_transactionreporting_regulation_report_excluded_cids
  WHERE cid IS NOT NULL
),
-- IsMifid filter from PRODUCTION SCD (10,526 IsMifid / 11,409 IsMifidByFCA as of 2026-06-10)
-- Source: main.regtech.gold_regreportdb_prod_dbo_reg_instruments_scd (exact SP source)
-- Uses proper SCD validity: ValidFrom <= report_date AND ValidTo > report_date
mifid_instruments AS (
  SELECT InstrumentID, IsMifid, IsMifidByFCA, IsFuture
  FROM main.regtech.gold_regreportdb_prod_dbo_reg_instruments_scd
  WHERE Tradable = true
    AND CAST(ValidFrom AS DATE) <= CAST('{report_date}' AS DATE)
    AND ValidTo > CAST('{report_date}' AS DATE)
),
-- CopyFund/FundType: SP uses staging mirror (MIFID2_ext_Mirror WHERE CopyFund=1)
-- Then LEFT JOINs Ext_TradeFund on ParentCID=FundAccountID for FundType only
mirror_copyfund AS (
  SELECT m.MirrorID, m.CopyFund,
         f.FundType AS FundTypeID
  FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_mirror m
  LEFT JOIN main.bi_db.bronze_etoro_trade_fund f ON m.ParentCID = f.FundAccountID
  WHERE m.CopyFund = 1
),
-- SP Block #1+4+5: EU report for CySEC (RegID=1), Seychelles (RegID=9), ME (RegID=11)
eu_report AS (
  SELECT t.*, 1 AS RegulationReportID FROM trade_pop t WHERE t.RegulationID IN (1, 9, 11)
),
-- SP Block #3: FCA→EU intercompany — FCA customers (RegID=2) ALSO reported to EU
fca_eu_intercompany AS (
  SELECT t.*, 1 AS RegulationReportID FROM trade_pop t WHERE t.RegulationID = 2
),
-- SP Block #2: UK report — ONLY FCA customers (RegID=2)
uk_report AS (
  SELECT t.*, 2 AS RegulationReportID FROM trade_pop t WHERE t.RegulationID = 2
),
all_reports AS (
  SELECT * FROM eu_report
  UNION ALL SELECT * FROM fca_eu_intercompany
  UNION ALL SELECT * FROM uk_report
),
report_final AS (
  SELECT
    ar.RegulationReportID,
    CAST(date_format(ar.ReportDate, 'yyyyMMdd') AS INT) AS DateID,
    ar.ReportDate, ar.CID, ar.RegulationID, ar.PositionID, ar.InstrumentID,
    ar.OpenORClose, ar.BuyORSell, ar.IDType,
    CASE WHEN ar.MirrorID IS NOT NULL AND ar.MirrorID > 0 THEN 1 ELSE 0 END AS IsCopy,
    COALESCE(mcf.CopyFund, 0) AS CopyFund,
    mcf.FundTypeID,
    'NEWT' AS ReportStatus,
    -- TRN: PositionID + suffix based on RegulationID and Report
    CONCAT(CAST(ar.PositionID AS STRING),
      CASE
        WHEN ar.RegulationReportID = 1 AND ar.RegulationID = 1 THEN ar.OpenORClose
        WHEN ar.RegulationReportID = 1 AND ar.RegulationID = 2 THEN CONCAT('UK', ar.OpenORClose)
        WHEN ar.RegulationReportID = 1 AND ar.RegulationID = 9 THEN CONCAT(ar.OpenORClose, 'SC', date_format(ar.ReportDate, 'yyyyMMdd'))
        WHEN ar.RegulationReportID = 1 AND ar.RegulationID = 11 THEN CONCAT(ar.OpenORClose, 'ME', date_format(ar.ReportDate, 'yyyyMMdd'))
        WHEN ar.RegulationReportID = 2 THEN ar.OpenORClose
        ELSE ar.OpenORClose
      END) AS TransactionReferenceNumber,
    '' AS TradingVenueTransactionIdentificationCode,
    -- ExecutingEntity: UK=eToro UK LEI, all others=eToro EU LEI
    CASE WHEN ar.RegulationReportID = 2 THEN '213800FLAB1OVA8OHT72'
         ELSE '213800GIFQMSV7HROS23'
    END AS ExecutingEntityIdentificationCode,
    'TRUE' AS InvestmentFirmCoveredBy201465EU,
    -------------------------------------------------------------------
    -- BUYER IDENTIFICATION
    -------------------------------------------------------------------
    -- BuyerIdentificationCodeType
    CASE
      -- FCA→EU intercompany: always LEI
      WHEN ar.RegulationReportID = 1 AND ar.RegulationID = 2 THEN 'LEI'
      -- Seychelles: always LEI
      WHEN ar.RegulationReportID = 1 AND ar.RegulationID = 9 THEN 'LEI'
      -- Standard EU RegID=1: customer side uses INT for natural persons
      WHEN ar.RegulationReportID = 1 AND ar.RegulationID = 1 THEN
        CASE WHEN ar.BuyORSell = 1 THEN  -- Customer is buyer
               CASE WHEN ar.IDType = 1 THEN 'INT' ELSE 'LEI' END
             ELSE 'LEI' END  -- Firm is buyer
      -- UK: customer side uses INT for natural persons
      WHEN ar.RegulationReportID = 2 THEN
        CASE WHEN ar.BuyORSell = 1 AND ar.IDType = 1 THEN 'INT' ELSE 'LEI' END
      ELSE 'LEI'
    END AS BuyerIdentificationCodeType,
    '' AS BuyerNPCode,
    -- BuyerIdentificationCode
    CASE
      -- FCA→EU intercompany: eToro UK buys, eToro EU sells (or reverse)
      WHEN ar.RegulationReportID = 1 AND ar.RegulationID = 2 THEN
        CASE WHEN ar.BuyORSell = 0 THEN '213800GIFQMSV7HROS23'  -- eToro EU is buyer
             ELSE '213800FLAB1OVA8OHT72' END  -- eToro UK is buyer
      -- Seychelles: eToro Seychelles or eToro EU
      WHEN ar.RegulationReportID = 1 AND ar.RegulationID = 9 THEN
        CASE WHEN ar.BuyORSell = 1 THEN '549300L7LPQNKJQ1IW32'  -- eToro Seychelles is buyer
             ELSE '213800GIFQMSV7HROS23' END  -- eToro EU is buyer
      -- Standard (EU RegID=1 or UK): customer vs eToro EU
      WHEN ar.BuyORSell = 1 THEN  -- Customer is buyer
        CASE WHEN ar.IDType <> 1 THEN COALESCE(ar.PIN_LEI, CAST(ar.CID AS STRING))
             ELSE CAST(ar.CID AS STRING) END
      ELSE '213800GIFQMSV7HROS23'  -- eToro EU is counterparty (both EU and UK reports)
    END AS BuyerIdentificationCode,
    -- BuyerCountryOfTheBranch
    CASE
      WHEN ar.RegulationReportID = 1 AND ar.RegulationID = 2 THEN
        CASE WHEN ar.BuyORSell = 1 THEN 'CY' ELSE '' END  -- FCA→EU: eToro UK buyer gets CY
      WHEN ar.RegulationReportID = 1 AND ar.RegulationID = 9 THEN
        CASE WHEN ar.BuyORSell = 1 THEN 'CY' ELSE '' END  -- Seychelles
      WHEN ar.RegulationReportID = 1 AND ar.BuyORSell = 1 THEN 'CY'  -- EU: customer is buyer → CY
      WHEN ar.RegulationReportID = 2 AND ar.BuyORSell = 1 THEN 'GB'  -- UK: customer is buyer → GB
      ELSE ''
    END AS BuyerCountryOfTheBranch,
    -- BuyerFirstNames/Surnames/DOB: only for standard customer flows
    CASE WHEN ar.BuyORSell = 1
              AND ar.RegulationReportID = 1 AND ar.RegulationID = 1
         THEN TRANSLATE(ar.FirstName, '{SPECIAL_CHARS}', '{REPLACEMENTS}')
         WHEN ar.BuyORSell = 1 AND ar.RegulationReportID = 2
         THEN TRANSLATE(ar.FirstName, '{SPECIAL_CHARS}', '{REPLACEMENTS}')
         ELSE '' END AS BuyerFirstNames,
    CASE WHEN ar.BuyORSell = 1
              AND ar.RegulationReportID = 1 AND ar.RegulationID = 1
         THEN TRANSLATE(ar.LastName, '{SPECIAL_CHARS}', '{REPLACEMENTS}')
         WHEN ar.BuyORSell = 1 AND ar.RegulationReportID = 2
         THEN TRANSLATE(ar.LastName, '{SPECIAL_CHARS}', '{REPLACEMENTS}')
         ELSE '' END AS BuyerSurnames,
    CASE WHEN ar.BuyORSell = 1
              AND ar.RegulationReportID = 1 AND ar.RegulationID = 1
         THEN CAST(ar.BirthDate AS STRING)
         WHEN ar.BuyORSell = 1 AND ar.RegulationReportID = 2
         THEN CAST(ar.BirthDate AS STRING)
         ELSE '' END AS BuyerDateOfBirth,
    -- BuyerDecisionMaker: populated for copy trades where customer is buyer (EU/UK standard only)
    CASE WHEN ar.BuyORSell = 1 AND ar.MirrorID > 0
              AND NOT (ar.RegulationReportID = 1 AND ar.RegulationID IN (2, 9))
         THEN 'LEI' ELSE '' END AS BuyerDecisionMakerCodeType,
    '' AS BuyerDecisionMakerNPCode,
    CASE WHEN ar.BuyORSell = 1 AND ar.MirrorID > 0
              AND NOT (ar.RegulationReportID = 1 AND ar.RegulationID IN (2, 9))
         THEN CASE WHEN ar.RegulationReportID = 2 THEN '213800FLAB1OVA8OHT72'
                   ELSE '213800GIFQMSV7HROS23' END
         ELSE '' END AS BuyerDecisionMakerCode,
    '' AS BuyerDecisionMakerFirstNames,
    '' AS BuyerDecisionMakerSurnames,
    '' AS BuyerDecisionMakerDateOfBirth,
    -------------------------------------------------------------------
    -- SELLER IDENTIFICATION
    -------------------------------------------------------------------
    -- SellerIdentificationCodeType
    CASE
      WHEN ar.RegulationReportID = 1 AND ar.RegulationID = 2 THEN 'LEI'  -- FCA→EU intercompany
      WHEN ar.RegulationReportID = 1 AND ar.RegulationID = 9 THEN 'LEI'  -- Seychelles
      WHEN ar.RegulationReportID = 1 AND ar.RegulationID = 1 THEN
        CASE WHEN ar.BuyORSell = 0 THEN  -- Customer is seller
               CASE WHEN ar.IDType = 1 THEN 'INT' ELSE 'LEI' END
             ELSE 'LEI' END
      WHEN ar.RegulationReportID = 2 THEN
        CASE WHEN ar.BuyORSell = 0 AND ar.IDType = 1 THEN 'INT' ELSE 'LEI' END
      ELSE 'LEI'
    END AS SellerIdentificationCodeType,
    '' AS SellerNPCode,
    -- SellerIdentificationCode
    CASE
      WHEN ar.RegulationReportID = 1 AND ar.RegulationID = 2 THEN
        CASE WHEN ar.BuyORSell = 1 THEN '213800GIFQMSV7HROS23'  -- eToro EU is seller
             ELSE '213800FLAB1OVA8OHT72' END  -- eToro UK is seller
      WHEN ar.RegulationReportID = 1 AND ar.RegulationID = 9 THEN
        CASE WHEN ar.BuyORSell = 0 THEN '549300L7LPQNKJQ1IW32'  -- eToro Seychelles is seller
             ELSE '213800GIFQMSV7HROS23' END  -- eToro EU is seller
      WHEN ar.BuyORSell = 0 THEN  -- Customer is seller
        CASE WHEN ar.IDType <> 1 THEN COALESCE(ar.PIN_LEI, CAST(ar.CID AS STRING))
             ELSE CAST(ar.CID AS STRING) END
      ELSE '213800GIFQMSV7HROS23'  -- eToro EU is counterparty
    END AS SellerIdentificationCode,
    -- SellerCountryOfTheBranch
    CASE
      WHEN ar.RegulationReportID = 1 AND ar.RegulationID = 2 THEN
        CASE WHEN ar.BuyORSell = 0 THEN 'CY' ELSE '' END
      WHEN ar.RegulationReportID = 1 AND ar.RegulationID = 9 THEN
        CASE WHEN ar.BuyORSell = 0 THEN 'CY' ELSE '' END
      WHEN ar.RegulationReportID = 1 AND ar.BuyORSell = 0 THEN 'CY'  -- EU: customer is seller → CY
      WHEN ar.RegulationReportID = 2 AND ar.BuyORSell = 0 THEN 'GB'  -- UK: customer is seller → GB
      ELSE ''
    END AS SellerCountryOfTheBranch,
    CASE WHEN ar.BuyORSell = 0
              AND ar.RegulationReportID = 1 AND ar.RegulationID = 1
         THEN TRANSLATE(ar.FirstName, '{SPECIAL_CHARS}', '{REPLACEMENTS}')
         WHEN ar.BuyORSell = 0 AND ar.RegulationReportID = 2
         THEN TRANSLATE(ar.FirstName, '{SPECIAL_CHARS}', '{REPLACEMENTS}')
         ELSE '' END AS SellerFirstNames,
    CASE WHEN ar.BuyORSell = 0
              AND ar.RegulationReportID = 1 AND ar.RegulationID = 1
         THEN TRANSLATE(ar.LastName, '{SPECIAL_CHARS}', '{REPLACEMENTS}')
         WHEN ar.BuyORSell = 0 AND ar.RegulationReportID = 2
         THEN TRANSLATE(ar.LastName, '{SPECIAL_CHARS}', '{REPLACEMENTS}')
         ELSE '' END AS SellerSurnames,
    CASE WHEN ar.BuyORSell = 0
              AND ar.RegulationReportID = 1 AND ar.RegulationID = 1
         THEN CAST(ar.BirthDate AS STRING)
         WHEN ar.BuyORSell = 0 AND ar.RegulationReportID = 2
         THEN CAST(ar.BirthDate AS STRING)
         ELSE '' END AS SellerDateOfBirth,
    -- SellerDecisionMaker
    CASE WHEN ar.BuyORSell = 0 AND ar.MirrorID > 0
              AND NOT (ar.RegulationReportID = 1 AND ar.RegulationID IN (2, 9))
         THEN 'LEI' ELSE '' END AS SellerDecisionMakerCodeType,
    '' AS SellerDecisionMakerNPCode,
    CASE WHEN ar.BuyORSell = 0 AND ar.MirrorID > 0
              AND NOT (ar.RegulationReportID = 1 AND ar.RegulationID IN (2, 9))
         THEN CASE WHEN ar.RegulationReportID = 2 THEN '213800FLAB1OVA8OHT72'
                   ELSE '213800GIFQMSV7HROS23' END
         ELSE '' END AS SellerDecisionMakerCode,
    '' AS SellerDecisionMakerFirstNames,
    '' AS SellerDecisionMakerSurnames,
    '' AS SellerDecisionMakerDateOfBirth,
    -------------------------------------------------------------------
    -- TRANSMISSION / DATETIME / CAPACITY
    -------------------------------------------------------------------
    CASE WHEN ar.RegulationReportID = 2 THEN 'true' ELSE 'false' END AS TransmissionOfOrderIndicator,
    '' AS TransmittingFirmIdentificationCodeForTheBuyer,
    '' AS TransmittingFirmIdentificationCodeForTheSeller,
    -- TradingDateTime: ISO 8601 with seconds=0 bumped to 1 (FCA rejects :00)
    CASE WHEN ar.OpenORClose = 'O' THEN
           CASE WHEN SECOND(ar.OpenOccurred) = 0
                THEN date_format(ar.OpenOccurred + INTERVAL 1 SECOND, "yyyy-MM-dd'T'HH:mm:ss'Z'")
                ELSE date_format(ar.OpenOccurred, "yyyy-MM-dd'T'HH:mm:ss'Z'") END
         ELSE
           CASE WHEN SECOND(ar.CloseOccurred) = 0
                THEN date_format(ar.CloseOccurred + INTERVAL 1 SECOND, "yyyy-MM-dd'T'HH:mm:ss'Z'")
                ELSE date_format(ar.CloseOccurred, "yyyy-MM-dd'T'HH:mm:ss'Z'") END
    END AS TradingDateTime,
    -- TradingCapacity: UK always AOTC, EU/SC/FCA-flow = DEAL (unless IsFuture=1 → AOTC)
    CASE WHEN ar.RegulationReportID = 2 THEN 'AOTC'
         WHEN inst.IsFuture = 1 THEN 'AOTC'
         ELSE 'DEAL'
    END AS TradingCapacity,
    'UNIT' AS QuantityType,
    CAST(ar.AmountInUnitsDecimal AS STRING) AS Quantity,
    '' AS QuantityCurrency,
    '' AS DerivativeNotionalIncreaseDecrease,
    -- PriceType: Indices (TypeID=4) = BSPS, else MNTR
    CASE WHEN inst.InstrumentTypeID = 4 THEN 'BSPS' ELSE 'MNTR' END AS PriceType,
    -- Price: Open=InitForexRate, Close=EndForexRate, GBX÷100
    CASE WHEN sc.Abbreviation = 'GBX'
         THEN CASE WHEN ar.OpenORClose = 'O' THEN CAST(ar.InitForexRate AS DOUBLE) ELSE CAST(ar.EndForexRate AS DOUBLE) END / 100.0
         ELSE CASE WHEN ar.OpenORClose = 'O' THEN CAST(ar.InitForexRate AS DOUBLE) ELSE CAST(ar.EndForexRate AS DOUBLE) END
    END AS Price,
    CASE WHEN sc.Abbreviation = 'GBX' THEN 'GBP'
         WHEN sc.Abbreviation = 'CNH' THEN 'CNY'
         ELSE SUBSTRING(sc.Abbreviation, 1, 3) END AS PriceCurrency,
    '' AS NetAmount,
    -- Venue: IsRealStockETF=1 → XOFF, else XXXX
    CASE WHEN ar.IsRealStockETF = 1 THEN 'XOFF' ELSE 'XXXX' END AS Venue,
    '' AS CountryOfTheBranchMembership,
    '' AS UpfrontPayment,
    '' AS UpfrontPaymentCurrency,
    '' AS ComplexTradeComponentId,
    -------------------------------------------------------------------
    -- INSTRUMENT FIELDS
    -------------------------------------------------------------------
    -- InstrumentIdentificationCode: ISIN for Real stocks/ETFs, empty for CFDs
    CASE WHEN ar.IsRealStockETF = 1 THEN COALESCE(im.ISINCode, '') ELSE '' END AS InstrumentIdentificationCode,
    -- InstrumentFullName: Real=empty, CFD=name+' CFD'
    CASE WHEN ar.IsRealStockETF = 1 THEN ''
         ELSE CONCAT(COALESCE(TRANSLATE(REPLACE(inst.InstrumentDisplayName, ',', ' '), '{SPECIAL_CHARS}', '{REPLACEMENTS}'), ''), ' CFD')
    END AS InstrumentFullName,
    -- InstrumentClassification: stub (complex branch-specific CFI logic deferred)
    CASE
      WHEN ar.IsRealStockETF = 1 THEN ''
      WHEN inst.InstrumentTypeID = 1 THEN 'JFTXCC'  -- Forex
      WHEN inst.InstrumentTypeID = 4 THEN 'JEIXCC'  -- Indices
      WHEN inst.InstrumentTypeID IN (5, 6) THEN 'JESXCC'  -- Stocks/ETF
      WHEN inst.InstrumentTypeID = 10 THEN 'JTMXCC'  -- Crypto
      WHEN inst.InstrumentTypeID = 2 THEN 'JTMXCC'  -- Commodities (simplified)
      ELSE ''
    END AS InstrumentClassification,
    -- NotionalCurrency1: sell currency for CFDs, empty for real
    CASE WHEN ar.IsRealStockETF = 1 THEN ''
         ELSE SUBSTRING(COALESCE(sc.Abbreviation, ''), 1, 3)
    END AS NotionalCurrency1,
    '' AS NotionalCurrency2,
    -- PriceMultiplier: 1 for non-futures
    CASE WHEN ar.IsRealStockETF = 1 THEN '' ELSE '1' END AS PriceMultiplier,
    -- UnderlyingInstrumentCode: ISIN for CFDs, empty for real
    CASE WHEN ar.IsRealStockETF = 1 THEN '' ELSE COALESCE(im.ISINCode, '') END AS UnderlyingInstrumentCode,
    -- UnderlyingIndexName: populated for indices (TypeID=4)
    CASE WHEN inst.InstrumentTypeID = 4
         THEN COALESCE(TRANSLATE(REPLACE(inst.InstrumentDisplayName, ',', ' '), '{SPECIAL_CHARS}', '{REPLACEMENTS}'), '')
         ELSE '' END AS UnderlyingIndexName,
    '' AS TermOfTheUnderlyingIndex,
    '' AS OptionType,
    '' AS StrikePriceType,
    '' AS StrikePrice,
    '' AS StrikePriceCurrency,
    '' AS OptionExerciseStyle,
    '' AS MaturityDate,
    '' AS ExpiryDate,
    -- DeliveryType: CASH for CFDs, empty for real
    CASE WHEN ar.IsRealStockETF = 1 THEN '' ELSE 'CASH' END AS DeliveryType,
    -------------------------------------------------------------------
    -- INVESTMENT DECISION / EXECUTION
    -------------------------------------------------------------------
    -- InvestmentDecisionWithinFirmType
    CASE
      WHEN ar.RegulationReportID = 1 AND ar.RegulationID = 9 THEN 'ALG'  -- Seychelles: always ALG
      WHEN ar.RegulationReportID = 2 AND (ar.MirrorID IS NULL OR ar.MirrorID = 0) THEN ''  -- UK non-copy: empty
      ELSE 'ALG'  -- EU standard + FCA→EU
    END AS InvestmentDecisionWithinFirmType,
    '' AS InvestmentDecisionWithinFirmNPCode,
    -- InvestmentDecisionWithinFirm
    CASE
      WHEN ar.RegulationReportID = 2 AND (ar.MirrorID IS NULL OR ar.MirrorID = 0) THEN ''  -- UK non-copy: empty
      WHEN ar.MirrorID IS NULL OR ar.MirrorID = 0 THEN 'ETOROBROKERAGE01'  -- Non-copy trades
      ELSE 'ETOROPM01'  -- Copy trades (simplified; FundType logic gated)
    END AS InvestmentDecisionWithinFirm,
    '' AS CountryOfTheBranchResponsibleForThePersonMakingTheInvestmentDecision,
    'ALG' AS ExecutionWithinFirmType,
    '' AS ExecutionWithinFirmNPCode,
    'ETOROBROKERAGE01' AS ExecutionWithinFirm,
    '' AS CountryOfTheBranchSupervisingThePersonResponsibleForTheExecution,
    '' AS WaiverIndicator,
    -------------------------------------------------------------------
    -- SHORT SELLING / INDICATORS
    -------------------------------------------------------------------
    -- ShortSellingIndicator: only for Stocks/ETF (TypeID 5,6) AND Real AND sell conditions
    CASE
      WHEN inst.InstrumentTypeID IN (5, 6) AND ar.IsRealStockETF = 1
           AND ar.BuyORSell = 0 AND ar.OpenORClose = 'C' THEN 'SELL'  -- seller is client, close
      WHEN inst.InstrumentTypeID IN (5, 6) AND ar.IsRealStockETF = 1
           AND ar.BuyORSell = 1 AND ar.OpenORClose = 'O' THEN 'SELL'  -- seller is eToro, open
      ELSE ''
    END AS ShortSellingIndicator,
    '' AS OTCPostTradeIndicator,
    -- CommodityDerivativeIndicator: 'false' for commodities (TypeID=2)
    CASE WHEN inst.InstrumentTypeID = 2 THEN 'false' ELSE '' END AS CommodityDerivativeIndicator,
    'false' AS SecuritiesFinancingTransactionIndicator,
    '' AS BranchLocation,
    '' AS TransactionType,
    '' AS LifecycleEvent,
    -- AssetClass: Indices/Stocks/ETF → Equity, else CurrencyType name
    CASE WHEN inst.InstrumentTypeID IN (4, 5, 6) THEN 'Equity'
         ELSE COALESCE(ct.AssetClassName, '') END AS AssetClass,
    ar.IsRealStockETF,
    CURRENT_TIMESTAMP() AS UpdateDate,
    0 AS BackReportingIndicator,
    0 AS RegChange
  FROM all_reports ar
  LEFT JOIN instruments inst ON ar.InstrumentID = inst.InstrumentID
  LEFT JOIN instrument_meta im ON ar.InstrumentID = im.InstrumentID
  LEFT JOIN sell_currency sc ON inst.SellCurrencyID = sc.CurrencyID
  LEFT JOIN currency_types ct ON inst.InstrumentTypeID = ct.CurrencyTypeID
  LEFT JOIN mirror_copyfund mcf ON ar.MirrorID = mcf.MirrorID
  WHERE ar.InstrumentID NOT IN (SELECT InstrumentID FROM excluded_instruments)
    AND ar.PositionID NOT IN (SELECT PositionID FROM excluded_positions)
    AND NOT (ar.RegulationReportID = 2 AND ar.CID IN (SELECT CID FROM excluded_cids))
    -- IsMifid instrument filter per SP logic:
    -- EU (RegID 1,9): IsMifidByESMA=1
    -- FCA→EU (RegID 2 into EU report): IsMifidByESMA=1 AND IsMifidByFCA=1
    -- UK (RegID 2): IsMifidByFCA=1
    AND EXISTS (
      SELECT 1 FROM mifid_instruments mi WHERE mi.InstrumentID = ar.InstrumentID
      AND (
        (ar.RegulationReportID = 1 AND ar.RegulationID = 2 AND mi.IsMifid = 1 AND mi.IsMifidByFCA = 1)
        OR (ar.RegulationReportID = 1 AND ar.RegulationID <> 2 AND mi.IsMifid = 1)
        OR (ar.RegulationReportID = 2 AND mi.IsMifidByFCA = 1)
      )
    )
)
SELECT * FROM report_final
"""

print("Building mifid2_report (SP-aligned v2)...")
mifid2_report_df = spark.sql(mifid2_report_sql)
cnt = mifid2_report_df.count()
print(f"mifid2_report: {cnt:,} rows, {len(mifid2_report_df.columns)} columns")

# Persist with external LOCATION
mifid2_report_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .option("path", "abfss://analysis@stgdpdlwe.dfs.core.windows.net/BI_OUTPUT/RegTechOps/mifid2_report") \
    .saveAsTable("main.regtech_ops_stg.bi_output_regtechops_mifid2_report")
print(f"\u2713 mifid2_report persisted ({cnt:,} rows)")

# Summary stats
print("\nBreakdown by RegulationReportID + OpenORClose:")
mifid2_report_df.groupBy("RegulationReportID", "OpenORClose").count().orderBy(1, 2).show()
print("\nBreakdown by RegulationReportID + RegulationID:")
mifid2_report_df.groupBy("RegulationReportID", "RegulationID").count().orderBy(1, 2).show()

In [0]:
%sql
-- Validation: Module 12 report output tables
SELECT 'mifid2_customer (output)' AS table_name, COUNT(*) AS row_count, COUNT(DISTINCT CID) AS distinct_cids
FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_customer
UNION ALL
SELECT 'mifid2_report_trade_population', COUNT(*), COUNT(DISTINCT CID)
FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_report_trade_population
UNION ALL
SELECT 'mifid2_report', COUNT(*), COUNT(DISTINCT CID)
FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_report
UNION ALL
SELECT 'mifid2_report (EU, ReportID=1)', COUNT(*), COUNT(DISTINCT CID)
FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_report WHERE RegulationReportID = 1
UNION ALL
SELECT 'mifid2_report (UK, ReportID=2)', COUNT(*), COUNT(DISTINCT CID)
FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_report WHERE RegulationReportID = 2

# Module 12: MIFID2 Report Output (Structural Test)

This notebook creates the final MIFID2_Report output tables for structural testing.

**Replicates SSIS package:** `SP_MIFID_Report` → `MIFID2_Report`, `MIFID2_ME_Report`, `MIFID2_Removed_OP_Partials`

**Pipeline:**
1. `mifid2_customer` (output) — enriches ext_customer with FTD, RegulationID mapping, report flags
2. `mifid2_report_trade_population` — position population with changelog, splits, partials
3. `mifid2_report` — final report with instrument metadata, pricing, buyer/seller projections

**Unresolved dependencies (gated/stubbed):**
- `fn_replacechar` UDF — special character conversion (stubbed as passthrough)
- `Dictionary.Ext_TradeFund` — copy fund mapping (stubbed as NULL)
- `MIFID2_Instruments_To_Exclude` — exclusion list (stubbed as empty)
- `InstrumentMetaData_SpecialChar_Conversion` — name cleanup (stubbed)
- PIN/UserAPI source — PII-blocked (stubbed as NULL in masked mode)
- `Reg_Ext_CustomerLatinName` — not found (stubbed)

**Targets created:**
- `bi_output_regtechops_mifid2_customer` — Customer output (enriched from ext)
- `bi_output_regtechops_mifid2_report_trade_population` — Intermediate position population
- `bi_output_regtechops_mifid2_report` — Main MIFID2 regulatory report